In [ ]:
import sys

!{sys.executable} -m pip install -U selenium webdriver-manager

In [ ]:
!python.exe -m pip install --upgrade pip
!pip install webdriver-manager selenium
!pip install selenium
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

In [ ]:
!pip install webdriver_manager
!pip install chromedriver_autoinstaller

In [34]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.chrome.options import Options

options = Options()
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

# 드라이버 설정
driver = webdriver.Chrome(options=options)
driver.get("https://map.kakao.com/")
print(driver.title)

카카오맵


In [ ]:
# csv 파일의 컬럼명 기준 검색 크롤링 -> 데이터프레임을 CSV로 저장
import re
import time
import pandas as pd

from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 텍스트 깔끔하게 추출
def clean_text(el):
    return (el.get_attribute("title") or el.text or "").strip()

# 검색 결과의 첫 번째 장소에서 주소를 가져오는 함수
def extract_first_address(driver, wait):
    first_item = wait.until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                "//*[@id='info.search.place.list']//li[contains(@class, 'PlaceItem')][1]"
            )
        )
    )
    # 도로명 주소
    address_el = first_item.find_element(By.CSS_SELECTOR, "p[data-id='address']")
    # 지번 주소
    lot_el = first_item.find_element(By.CSS_SELECTOR, "p[data-id='otherAddr']")

    re_address = clean_text(address_el)
    re_num = clean_text(lot_el)

    # text로 가져왔을 때 \"(지번) 일도일동 1425-3\" 형태일 수 있으므로 제거
    re_num = re.sub(r\"^\\(지번\\)\\s*\", \"\", re_num).strip()

    return re_address, re_num

re_df = pd.read_csv(\"finaltest.csv\")
re_df[\"re_address\"] = \"\"
re_df[\"re_num\"] = \"\"
wait = WebDriverWait(driver, 10)

driver.get(\"https://map.kakao.com/\")

for i, merc_nm in re_df[\"merc_nm\"].fillna(\"\").items():
    # 검색 정확도 높이기 위해 지역명 앞에 붙임
    keyword = f\"제주 {merc_nm}\".strip()

    try:
        # search.keyword.query = ID
        search_box = wait.until(
            EC.presence_of_element_located((By.ID, \"search.keyword.query\"))
        )

        search_box.clear()
        search_box.send_keys(keyword)
        search_box.send_keys(Keys.ENTER)

        # 검색 결과 로딩 대기
        time.sleep(1.2)

        re_address, re_num = extract_first_address(driver, wait)

        re_df.at[i, \"re_address\"] = re_address
        re_df.at[i, \"re_num\"] = re_num

        #print(i, keyword, re_address, re_num)
        
    # 실패 로그 출력
    except Exception as e:
        re_df.at[i, \"re_address\"] = \"\"
        re_df.at[i, \"re_num\"] = \"\"
        print(f\"[실패] {i} / {keyword} / {e}\")

    # 너무 빠른 요청 방지
    time.sleep(0.7)
    
#csv 파일 새로 저장
re_df.to_csv(\"제주_카카오맵_주소추출.csv\", index=False, encoding=\"utf-8-sig\")

In [ ]:
import sys

!{sys.executable} -m pip install -U pandas

In [ ]:
# 리스트 -> 주소 키워드 -> 서울 + 주소 키워드로 검색 -> 크롤링
import re
import time
import pandas as pd

from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 텍스트 깔끔하게 추출
def clean_text(el):
    return (el.get_attribute(\"title\") or el.text or \"\").strip()

# 검색 결과의 첫 번째 장소에서 주소를 가져오는 함수
def extract_first_address(driver, wait):
    first_item = wait.until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                \"//*[@id='info.search.place.list']//li[contains(@class, 'PlaceItem')][1]\"
            )
        )
    )
    # 도로명 주소
    address_el = first_item.find_element(By.CSS_SELECTOR, \"p[data-id='address']\")
    # 지번 주소
    lot_el = first_item.find_element(By.CSS_SELECTOR, \"p[data-id='otherAddr']\")

    re_address = clean_text(address_el)
    re_num = clean_text(lot_el)

    # text로 가져왔을 때 \"(지번) 일도일동 1425-3\" 형태일 수 있으므로 제거
    re_num = re.sub(r\"^\\(지번\\)\\s*\", \"\", re_num).strip()

    return re_address, re_num

addresses = [\"아지트\", \"집\", \"당근\", \"토끼굴\", \"바나나\", \"오리농장\", \"커프\", \"비나레스토랑\", \"성심당\"]
re_df = pd.DataFrame({\"address_keyword\": addresses})

re_df[\"re_address\"] = \"\"
re_df[\"re_num\"] = \"\"
wait = WebDriverWait(driver, 10)

driver.get(\"https://map.kakao.com/\")

for i, keyword in re_df[\"address_keyword\"].fillna(\"\").items():
    # 검색 정확도 높이기 위해 지역명 앞에 붙임
    keyword = f\"서울 {keyword}\".strip()

    try:
        # search.keyword.query = ID
        search_box = wait.until(
            EC.presence_of_element_located((By.ID, \"search.keyword.query\"))
        )

        search_box.clear()
        search_box.send_keys(keyword)
        search_box.send_keys(Keys.ENTER)

        # 검색 결과 로딩 대기
        time.sleep(1.2)

        re_address, re_num = extract_first_address(driver, wait)

        re_df.at[i, \"re_address\"] = re_address
        re_df.at[i, \"re_num\"] = re_num

        #print(i, keyword, re_address, re_num)
        
    # 실패 로그 출력
    except Exception as e:
        print(f\"[실패] {keyword} / {e}\")

    # 너무 빠른 요청 방지
    time.sleep(1)
print(re_df.head(10))